<a href="https://colab.research.google.com/github/Kemal1101/Model-Comparison-for-Sistem-Tes-Minat-Career/blob/main/Forward_%26_Backward_Chaining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import sys
from itertools import permutations
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════
#  1. MEMUAT DATASET
# ══════════════════════════════════════════════════════════════════
try:
    # Path untuk dataset occupations (sesuaikan dengan path di Colab Anda)
    df_occupations = pd.read_excel('/content/drive/MyDrive/Kuis_SBP/Onet_Top3_Occupations_by_InterestCode.xlsx')

    # Path untuk dataset jawaban kuesioner dari Google Form
    # Pastikan file ini sudah di-upload ke Colab atau sesuaikan path Drive-nya
    file_jawaban = '/content/drive/MyDrive/Kuis_SBP/Preprocessed_RIASEC_Dataset.csv'
    df_responses = pd.read_csv(file_jawaban)

    print("Knowledge Base & Dataset Jawaban berhasil dimuat!")
except FileNotFoundError as e:
    print(f"Error: File tidak ditemukan! {e}")
    sys.exit(1)
except Exception as e:
    print(f"Error tidak terduga: {e}")
    sys.exit(1)

# Mapping Kategori untuk 30 Pertanyaan (Masing-masing 5 soal berurutan)
# Q1-Q5(R), Q6-10(I), Q11-15(A), Q16-20(S), Q21-25(E), Q26-30(C)
KATEGORI_SOAL = (['R']*5) + (['I']*5) + (['A']*5) + (['S']*5) + (['E']*5) + (['C']*5)


# ══════════════════════════════════════════════════════════════════
#  2. KOMPONEN INFERENCE ENGINE
# ══════════════════════════════════════════════════════════════════
class WorkingMemory:
    def __init__(self, fakta_awal: dict):
        self._fakta = dict(fakta_awal)
        self._log   = []

    def ada(self, kunci):
        return kunci in self._fakta

    def get(self, kunci, default=None):
        return self._fakta.get(kunci, default)

    def tambah(self, kunci, nilai, sumber="sistem"):
        if kunci not in self._fakta:
            self._fakta[kunci] = nilai
            self._log.append((kunci, nilai, sumber))
            return True
        return False

class Rule:
    def __init__(self, nama: str, kondisi, aksi, keterangan: str):
        self.nama       = nama
        self.kondisi    = kondisi
        self.aksi       = aksi
        self.keterangan = keterangan

    def dapat_dijalankan(self, wm: WorkingMemory) -> bool:
        return self.kondisi(wm)

    def eksekusi(self, wm: WorkingMemory) -> dict:
        return self.aksi(wm)

class ForwardChainingEngine:
    def __init__(self, rule_base: list, df_occupations):
        self.rule_base      = rule_base
        self.df_occupations = df_occupations

    def jalankan(self, jawaban_user: list, verbose: bool = True):
        wm       = WorkingMemory({'jawaban_user': jawaban_user})
        siklus   = 0
        trace    = []
        diproses = set()

        if verbose:
            self._log_header(len(jawaban_user))

        while True:
            siklus += 1
            agenda = [
                r for r in self.rule_base
                if r.nama not in diproses and r.dapat_dijalankan(wm)
            ]

            if verbose:
                print(f"\n{'─'*54}")
                print(f"[SIKLUS {siklus}] MATCH  → {len(agenda)} rule aktif: {[r.nama for r in agenda]}")

            if not agenda:
                if verbose:
                    print(f"[SIKLUS {siklus}] Agenda kosong → inference selesai.")
                break

            rule_dipilih = agenda[0]
            if verbose:
                print(f"[SIKLUS {siklus}] SELECT → {rule_dipilih.nama}\n             Aturan : {rule_dipilih.keterangan}")

            fakta_baru = rule_dipilih.eksekusi(wm)
            diproses.add(rule_dipilih.nama)

            fakta_ditambahkan = []
            for kunci, nilai in fakta_baru.items():
                if wm.tambah(kunci, nilai, sumber=rule_dipilih.nama):
                    fakta_ditambahkan.append((kunci, nilai))

            if verbose:
                print(f"[SIKLUS {siklus}] EXECUTE → Fakta baru yang dihasilkan:")
                for k, v in fakta_ditambahkan:
                    ringkas = self._ringkas_nilai(v)
                    print(f"             + {k} = {ringkas}")

            trace.append({'siklus': siklus, 'rule': rule_dipilih.nama, 'fakta': fakta_ditambahkan})

        if verbose:
            self._tampilkan_kesimpulan(wm)

        return wm, trace

    def _log_header(self, n_jawaban: int):
        print("╔══════════════════════════════════════════════════════╗")
        print("║      MESIN INFERENSI FORWARD CHAINING - RIASEC       ║")
        print("╚══════════════════════════════════════════════════════╝")
        print(f"\n[INIT] Fakta Awal  → jawaban_user[0..{n_jawaban - 1}] (Skala Likert 1-5)")
        print(f"[INIT] Rule Base   → {len(self.rule_base)} rules terdaftar")

    def _ringkas_nilai(self, nilai) -> str:
        if isinstance(nilai, list): return f"[{len(nilai)} item]"
        return str(nilai)

    def _tampilkan_kesimpulan(self, wm: WorkingMemory):
        rekomendasi = wm.get('rekomendasi_profesi', [])
        print("\n" + "═" * 54)
        print("  KESIMPULAN AKHIR")
        print("═" * 54)
        print(f"  Skor RIASEC    : {wm.get('skor_riasec', {})}")
        print(f"  Kode Holland   : [{wm.get('kode_holland', '?')}]")
        print(f"  Metode Match   : {wm.get('level_match', 'Tidak ditemukan')}")
        print("─" * 54)
        if rekomendasi:
            for i, row in enumerate(rekomendasi, start=1):
                print(f"  {i:2}. {row['Occupation']} (Kode: {row['Interest Code']} | Job Zone: {row['Job Zone']})")
        else:
            print("  Tidak ditemukan profesi yang cocok.")
        print("═" * 54)


# ══════════════════════════════════════════════════════════════════
#  3. RULE BASE - Definisi Aturan IF-THEN
# ══════════════════════════════════════════════════════════════════
def _r1_kondisi(wm: WorkingMemory) -> bool: return wm.ada('jawaban_user') and not wm.ada('skor_riasec')
def _r1_aksi(wm: WorkingMemory) -> dict:
    # UPDATE LIKERT: Tambahkan nilai 1-5 langsung ke skor
    skor = {'R': 0, 'I': 0, 'A': 0, 'S': 0, 'E': 0, 'C': 0}
    for i, jawaban in enumerate(wm.get('jawaban_user')):
        kat = KATEGORI_SOAL[i]
        skor[kat] += jawaban
    return {'skor_riasec': skor}

def _r2_kondisi(wm: WorkingMemory) -> bool: return wm.ada('skor_riasec') and not wm.ada('kode_holland')
def _r2_aksi(wm: WorkingMemory) -> dict:
    skor_urut = sorted(wm.get('skor_riasec').items(), key=lambda x: x[1], reverse=True)
    kode      = ''.join(item[0] for item in skor_urut[:3])
    return {'kode_holland': kode, 'detail_skor': skor_urut}

def _r3_kondisi(wm: WorkingMemory) -> bool: return wm.ada('kode_holland') and not wm.ada('kandidat_l1')
def _r3_aksi(wm: WorkingMemory) -> dict:
    kode3 = wm.get('kode_holland')[:3]
    hasil = df_occupations[df_occupations['Interest Code'].astype(str).str.startswith(kode3, na=False)]
    return {'kandidat_l1': hasil.to_dict('records')}

def _r4_kondisi(wm: WorkingMemory) -> bool: return wm.ada('kandidat_l1') and len(wm.get('kandidat_l1', [])) < 10 and not wm.ada('kandidat_l2')
def _r4_aksi(wm: WorkingMemory) -> dict:
    kode3  = wm.get('kode_holland')[:3]
    l1_set = {r['Occupation'] for r in wm.get('kandidat_l1', [])}
    perms  = [''.join(p) for p in permutations(kode3) if ''.join(p) != kode3]
    hasil_l2 = []
    for perm in perms:
        df_perm = df_occupations[df_occupations['Interest Code'].astype(str).str.startswith(perm, na=False)]
        for _, row in df_perm.iterrows():
            if row['Occupation'] not in l1_set:
                hasil_l2.append(row.to_dict())
                l1_set.add(row['Occupation'])
    return {'kandidat_l2': hasil_l2}

def _r5_kondisi(wm: WorkingMemory) -> bool: return (wm.ada('kandidat_l1') or wm.ada('kandidat_l2')) and not wm.ada('rekomendasi_profesi')
def _r5_aksi(wm: WorkingMemory) -> dict:
    l1 = wm.get('kandidat_l1', [])
    l2 = wm.get('kandidat_l2', [])
    gabungan = (l1 + l2)[:10]
    kode3 = wm.get('kode_holland', '')[:3]
    parts = []
    if l1: parts.append(f"Level 1: {len(l1)} profesi (exact: {kode3})")
    if l2: parts.append(f"Level 2: {len(l2)} profesi (permutasi)")
    return {'rekomendasi_profesi': gabungan, 'level_match': " | ".join(parts) if parts else "Tidak ditemukan"}

RULE_BASE = [
    Rule("R1", _r1_kondisi, _r1_aksi, "IF jawaban_user ada       → THEN hitung skor per kategori RIASEC"),
    Rule("R2", _r2_kondisi, _r2_aksi, "IF skor_riasec ada        → THEN tentukan kode Holland 3-huruf"),
    Rule("R3", _r3_kondisi, _r3_aksi, "IF kode_holland ada       → THEN cari profesi Level 1 (exact match)"),
    Rule("R4", _r4_kondisi, _r4_aksi, "IF Level-1 < 10           → THEN cari profesi Level 2 (semua permutasi)"),
    Rule("R5", _r5_kondisi, _r5_aksi, "IF kandidat_l1/l2 ada    → THEN gabungkan & finalisasi rekomendasi"),
]

# ══════════════════════════════════════════════════════════════════
#  4. PENGUJIAN DENGAN DATA ASLI
# ══════════════════════════════════════════════════════════════════

# Pilih salah satu responden untuk diuji (misal: index 0 untuk Kayla)
responden_idx = 0
data_responden = df_responses.iloc[responden_idx]
nama = data_responden['Nama Lengkap']

# Mengambil jawaban kuesioner dari kolom indeks ke-5 sampai ke-34 (Total 30 Kolom)
jawaban_asli = data_responden.iloc[5:35].astype(int).tolist()

print("\n" + "="*50)
print(f"MENGANALISIS DATA RESPONDEN: {nama}")
print(f"Jawaban Mentah (1-5): {jawaban_asli}")
print("="*50)

# Jalankan Forward Chaining Engine
engine = ForwardChainingEngine(RULE_BASE, df_occupations)
wm, trace = engine.jalankan(jawaban_asli, verbose=True)

Mounted at /content/drive
Knowledge Base & Dataset Jawaban berhasil dimuat!

MENGANALISIS DATA RESPONDEN: Kayla Shabina Putri Giani
Jawaban Mentah (1-5): [1, 2, 3, 3, 2, 4, 4, 3, 3, 4, 2, 1, 3, 1, 4, 4, 4, 3, 3, 5, 4, 3, 3, 2, 2, 3, 3, 3, 3, 4]
╔══════════════════════════════════════════════════════╗
║      MESIN INFERENSI FORWARD CHAINING - RIASEC       ║
╚══════════════════════════════════════════════════════╝

[INIT] Fakta Awal  → jawaban_user[0..29] (Skala Likert 1-5)
[INIT] Rule Base   → 5 rules terdaftar

──────────────────────────────────────────────────────
[SIKLUS 1] MATCH  → 1 rule aktif: ['R1']
[SIKLUS 1] SELECT → R1
             Aturan : IF jawaban_user ada       → THEN hitung skor per kategori RIASEC
[SIKLUS 1] EXECUTE → Fakta baru yang dihasilkan:
             + skor_riasec = {'R': 11, 'I': 18, 'A': 11, 'S': 19, 'E': 14, 'C': 16}

──────────────────────────────────────────────────────
[SIKLUS 2] MATCH  → 1 rule aktif: ['R2']
[SIKLUS 2] SELECT → R2
             Aturan : IF

In [ ]:
import pandas as pd
import sys
from itertools import permutations
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# ══════════════════════════════════════════════════════════════════
#  1. MEMUAT DATASET & MAPPING
# ══════════════════════════════════════════════════════════════════
try:
    df_occupations = pd.read_excel('/content/drive/MyDrive/Kuis_SBP/Onet_Top3_Occupations_by_InterestCode.xlsx')
    file_jawaban = '/content/drive/MyDrive/Kuis_SBP/Preprocessed_RIASEC_Dataset.csv'
    df_responses = pd.read_csv(file_jawaban)
    print("Dataset Profesi & Jawaban berhasil dimuat!")
except Exception as e:
    print(f"Error: {e}")
    sys.exit(1)

# Mapping 30 Pertanyaan berurutan ke kategori RIASEC
KATEGORI_SOAL = (['R']*5) + (['I']*5) + (['A']*5) + (['S']*5) + (['E']*5) + (['C']*5)

def tentukan_kode_holland(skor_riasec):
    """Fungsi helper untuk mendapatkan 3 huruf dominan."""
    skor_urut = sorted(skor_riasec.items(), key=lambda item: item[1], reverse=True)
    kode_dominan = "".join([item[0] for item in skor_urut[:3]])
    return kode_dominan, skor_urut


# ══════════════════════════════════════════════════════════════════
#  2. MESIN INFERENSI BACKWARD CHAINING
# ══════════════════════════════════════════════════════════════════
class BackwardChainingEngine:
    def __init__(self, df_occupations):
        self.df_occupations = df_occupations

    def buktikan(self, target_profesi: str, jawaban_user: list):
        self._trace = []
        self._wm = {}
        self._jawaban = jawaban_user
        self._depth = 0

        self._log("╔══════════════════════════════════════════════════════╗")
        self._log("║       MESIN INFERENSI BACKWARD CHAINING - RIASEC    ║")
        self._log("╚══════════════════════════════════════════════════════╝")
        self._log(f"\nGOAL : Buktikan → cocok(user, '{target_profesi}')\n")

        terbukti, info = self._goal_cocok(target_profesi)
        return terbukti, self._trace, info

    def _goal_cocok(self, target_profesi: str):
        self._log(f"{'─'*54}")
        self._log(f"[GOAL] Buktikan: cocok(user, '{target_profesi}')")
        self._depth += 1

        self._log(f"\n[F1] Mencari kode_profesi('{target_profesi}') di database O*NET ...")
        profesi_row = self._lookup_profesi(target_profesi)

        if profesi_row is None:
            self._log(f"[✗] GAGAL: Profesi '{target_profesi}' tidak ditemukan.")
            self._depth -= 1
            return False, {'error': 'Tidak ditemukan', 'target': target_profesi, 'terbukti': False, 'level': None}

        nama_profesi = profesi_row['Occupation']
        kode_profesi = str(profesi_row['Interest Code'])[:3].upper()
        job_zone     = profesi_row.get('Job Zone', '-')
        self._log(f"[✓] F1 terpenuhi: kode_profesi('{nama_profesi}') = [{kode_profesi}]  |  Job Zone: {job_zone}")

        self._log(f"\n[SUB-GOAL] Turunkan kode_user(user) via R5 ← R6 ← F2")
        self._depth += 1
        kode_user, skor_user = self._goal_kode_user()
        self._depth -= 1

        self._log(f"\n[R1] Terapkan: kode_cocok_exact([{kode_user[:3]}], [{kode_profesi}])")
        self._depth += 1
        terbukti_r1 = self._goal_kode_cocok_exact(kode_user, kode_profesi)
        self._depth -= 1

        if terbukti_r1:
            level, terbukti = "Level 1 - Exact Match", True
            self._log(f"\n[✓] GOAL TERBUKTI  ←  via R1 (Exact Match)")
        else:
            self._log(f"\n[R2] Terapkan: kode_cocok_permutasi([{kode_user[:3]}], [{kode_profesi}])")
            self._depth += 1
            terbukti_r2 = self._goal_kode_cocok_permutasi(kode_user, kode_profesi)
            self._depth -= 1

            if terbukti_r2:
                level, terbukti = "Level 2 - Permutasi", True
                self._log(f"\n[✓] GOAL TERBUKTI  ←  via R2 (Permutasi)")
            else:
                level, terbukti = None, False
                self._log(f"\n[✗] GOAL TIDAK TERBUKTI: Tidak ada rule yang terpenuhi.")

        self._depth -= 1
        return terbukti, {'target': nama_profesi, 'kode_profesi': kode_profesi, 'job_zone': job_zone,
                          'skor_user': skor_user, 'kode_user': kode_user, 'terbukti': terbukti, 'level': level}

    def _goal_kode_user(self):
        if 'kode_user' in self._wm:
            self._log(f"[WM] kode_user sudah ada di working memory: [{self._wm['kode_user']}]")
            return self._wm['kode_user'], self._wm['skor_riasec']

        self._log(f"[R5] Menurunkan: kode_user ← top3_kategori(skor_riasec)")
        self._depth += 1
        skor = self._goal_skor_riasec()
        kode, detail = tentukan_kode_holland(skor)
        urutan_skor = {k: v for k, v in detail}
        self._log(f"[R5] top3_kategori({urutan_skor}) → kode_user = [{kode}]")
        self._depth -= 1

        self._wm['kode_user'], self._wm['skor_riasec'] = kode, skor
        return kode, skor

    def _goal_skor_riasec(self):
        if 'skor_riasec' in self._wm: return self._wm['skor_riasec']

        # UPDATE LIKERT: Menjumlahkan bobot jawaban (1-5)
        self._log(f"[R6] Menurunkan: skor_riasec ← Σ akumulasi nilai jawaban (1-5) per kategori")
        self._log(f"[F2] Membaca fakta dasar: 30 jawaban user")

        skor = {'R': 0, 'I': 0, 'A': 0, 'S': 0, 'E': 0, 'C': 0}
        for i, jawaban in enumerate(self._jawaban):
            kat = KATEGORI_SOAL[i]
            skor[kat] += jawaban

        self._log(f"[R6] skor_riasec = {skor}")
        self._wm['skor_riasec'] = skor
        return skor

    def _goal_kode_cocok_exact(self, kode_user, kode_profesi):
        ku3 = kode_user[:3]
        if ku3 == kode_profesi:
            self._log(f"[R3] [{ku3}] == [{kode_profesi}]  →  ✓ TERPENUHI")
            return True
        self._log(f"[R3] [{ku3}] ≠  [{kode_profesi}]  →  ✗ TIDAK TERPENUHI")
        return False

    def _goal_kode_cocok_permutasi(self, kode_user, kode_profesi):
        ku3_sorted = sorted(kode_user[:3])
        kp_sorted  = sorted(kode_profesi)
        semua_perm = [''.join(p) for p in permutations(kode_profesi)]

        self._log(f"[R4] sorted([{kode_user[:3]}])={ku3_sorted}  vs  sorted([{kode_profesi}])={kp_sorted}")
        self._log(f"[R4] Semua permutasi dari [{kode_profesi}]: {semua_perm}")

        if ku3_sorted == kp_sorted:
            cocok_perm = next(p for p in semua_perm if p == kode_user[:3])
            self._log(f"[R4] [{kode_user[:3]}] cocok dengan permutasi [{cocok_perm}]  →  ✓ TERPENUHI")
            return True
        self._log(f"[R4] Tidak ada permutasi yang cocok  →  ✗ TIDAK TERPENUHI")
        return False

    def _lookup_profesi(self, target):
        row = self.df_occupations[self.df_occupations['Occupation'].str.lower() == target.lower()]
        if not row.empty: return row.iloc[0]
        row = self.df_occupations[self.df_occupations['Occupation'].str.lower().str.contains(target.lower(), na=False)]
        if not row.empty: return row.iloc[0]
        return None

    def _log(self, msg):
        self._trace.append(f"{'  ' * self._depth}{msg}")


# ══════════════════════════════════════════════════════════════════
#  3. PENGUJIAN DENGAN DATA ASLI (SKALA LIKERT)
# ══════════════════════════════════════════════════════════════════

# Pilih salah satu responden (misal index 1: Aidatul Rosida)
responden_idx = 1
data_responden = df_responses.iloc[responden_idx]
nama_user = data_responden['Nama Lengkap']
jawaban_asli = data_responden.iloc[5:35].astype(int).tolist()

# [Langkah Bantuan] Hitung manual kode responden untuk mencari profesi target (GOAL)
skor_awal = {'R': 0, 'I': 0, 'A': 0, 'S': 0, 'E': 0, 'C': 0}
for i, ans in enumerate(jawaban_asli): skor_awal[KATEGORI_SOAL[i]] += ans
kode_awal, _ = tentukan_kode_holland(skor_awal)
kode3 = kode_awal[:3]

# [Langkah Bantuan] Cari profesi dari O*NET untuk diuji berdasarkan kode user
p_l1 = df_occupations[df_occupations['Interest Code'].astype(str).str.startswith(kode3, na=False)]['Occupation'].tolist()
semua_perm = [''.join(p) for p in permutations(kode3) if ''.join(p) != kode3]
p_l2 = [p for perm in semua_perm for p in df_occupations[df_occupations['Interest Code'].astype(str).str.startswith(perm, na=False)]['Occupation'].tolist()]
p_tidak = df_occupations[~df_occupations['Occupation'].isin(set(p_l1 + p_l2))]['Occupation'].tolist()

# Pilih 4 Goal Target (1 Exact, 1 Permutasi, 2 Tidak Cocok) untuk mensimulasikan sistem Pakar
goals_l1 = p_l1[:1] if p_l1 else []
goals_l2 = p_l2[:1] if p_l2 else []
goals_tidak = p_tidak[:2]
target_professions = goals_l1 + goals_l2 + goals_tidak

# ══════════════════════════════════════════════════════════════════
#  4. JALANKAN PEMBUKTIAN & TAMPILKAN HASIL
# ══════════════════════════════════════════════════════════════════
engine = BackwardChainingEngine(df_occupations)

WIDTH = 60
print("\n" + "═" * WIDTH)
print("   SIMULASI BACKWARD CHAINING - LIKERT SCALE (1-5)")
print("═" * WIDTH)
print(f"   Nama User        : {nama_user}")
print(f"   Total soal       : 30 Jawaban Likert")
print(f"   Kode Holland user: [{kode3}]  (skor: {skor_awal})")
print(f"   Profesi diuji    : {len(target_professions)}")
print("═" * WIDTH)

ringkasan = []
for target in target_professions:
    terbukti, trace, info = engine.buktikan(target, jawaban_asli)
    print()
    for line in trace: print(line)

    status = "COCOK ✓" if terbukti else "TIDAK COCOK ✗"
    level  = info.get('level', '-') or 'Tidak ada rule cocok'
    ringkasan.append((info.get('target', target), info.get('kode_profesi', '?'), info.get('kode_user', '?'), level, status))

print("\n" + "═" * 70)
print("   RINGKASAN HASIL PEMBUKTIAN BACKWARD CHAINING")
print("═" * 70)
print(f"   {'Profesi':<35} {'K.Profesi':^10} {'K.User':^8} {'Hasil'}")
print("─" * 70)
for nama, kp, ku, level, status in ringkasan:
    nama_potong = str(nama)[:34]
    print(f"   {nama_potong:<35} [{kp:^9}] [{ku[:3]:^6}]  {status}")
    print(f"   {'':35}   Metode: {level}")
print("═" * 70)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Dataset Profesi & Jawaban berhasil dimuat!

════════════════════════════════════════════════════════════
   SIMULASI BACKWARD CHAINING - LIKERT SCALE (1-5)
════════════════════════════════════════════════════════════
   Nama User        : aidatul rosida
   Total soal       : 30 Jawaban Likert
   Kode Holland user: [ASE]  (skor: {'R': 15, 'I': 18, 'A': 23, 'S': 21, 'E': 19, 'C': 19})
   Profesi diuji    : 4
════════════════════════════════════════════════════════════

╔══════════════════════════════════════════════════════╗
║       MESIN INFERENSI BACKWARD CHAINING - RIASEC    ║
╚══════════════════════════════════════════════════════╝

GOAL : Buktikan → cocok(user, 'Actors')

──────────────────────────────────────────────────────
[GOAL] Buktikan: cocok(user, 'Actors')
  
[F1] Mencari kode_profesi('Actors') di database O*NET ...
  [✓] F1 terpenuhi: kode_profesi